# LLM


## Import Data

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Load the CSV file into a DataFrame with the correct encoding
decentraland= pd.read_csv('https://raw.githubusercontent.com/Xintong1122/KDD/main/Data/Decentraland_general.csv', encoding='ISO-8859-1')

# Assume decentraland is the DataFrame you want to remove unnamed columns from
decentraland = decentraland.loc[:, ~decentraland.columns.str.startswith('Unnamed:')]

In [ ]:
# Date format change
decentraland['Date'] = pd.to_datetime(decentraland['Date'], format='%Y-%m-%dT%H:%M:%S.%f%z')
decentraland['Date'] = decentraland['Date'].dt.strftime('%m/%d/%Y %I:%M %p')

In [ ]:
decentraland.head()

,AuthorID,Author,Date,Content,Attachments,Reactions
0,1.460000e+17,chartguy.,01/01/2024 05:56 AM,Hi,NaN,NaN
1,1.020000e+18,Decentraland#7255,01/01/2024 08:00 AM,NaN,NaN,NaN
2,1.030000e+18,toxicwaifu.dcl.eth,01/01/2024 08:01 AM,Happy New Year from Germany â¨ð,NaN,NaN
3,8.740000e+17,cheddarqueso,01/01/2024 08:23 AM,Happy New Year DCL fam and team. I am thankful...,NaN,"ð (2),â¤ï¸ (1)"
4,8.580000e+17,help_center017,01/01/2024 12:11 PM,Welcome,NaN,NaN


In [ ]:
decentraland.tail()

,AuthorID,Author,Date,Content,Attachments,Reactions
2066,3.500000e+17,.vitsky,06/13/2024 11:17 PM,You can buy lands on our Marketplace ð http...,NaN,NaN
2067,7.810000e+17,eldanak_dcl,06/14/2024 01:24 AM,# ð¥ **Feature Focus Thursday** ð¥\n\nWe w...,NaN,ð¥ (3)
2068,1.020000e+18,Decentraland#7255,06/14/2024 08:00 AM,NaN,NaN,NaN
2069,3.990000e+17,cybermike.,06/14/2024 04:51 PM,@Kaze_no_Kai scam,NaN,"â¤ï¸ (3),ð (3)"
2070,1.020000e+18,Decentraland#7255,06/15/2024 08:00 AM,NaN,NaN,NaN


## Text Processing

In [ ]:
# Delete blank rows in the "Content" column.
decentraland_cleaned = decentraland.dropna(subset=['Content'])

# Save the cleaned DataFrame to a new CSV file
decentraland_cleaned.to_csv('message_cleaned.csv', index=False)


# Sentiment Analysis using LLM

## Sentiment Label and Sentiment Score

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Read CSV file from GitHub
url = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/Data/message_cleaned.csv'
df = pd.read_csv(url)

# Extract data from "Content" column
content_data = df['Content'].tolist()

# Load the tokenizer
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Maximum sequence length for the model
max_length = 512

# Filter out texts longer than max_length
filtered_content_data = []
filtered_indices = []

for i, text in enumerate(content_data):
    tokens = tokenizer.encode(text, max_length=max_length, truncation=True)
    if len(tokens) <= max_length:
        filtered_content_data.append(text)
        filtered_indices.append(i)
    print(f"Text length: {len(tokens)}, included: {len(tokens) <= max_length}")

# Create a new DataFrame with the filtered data
filtered_df = df.iloc[filtered_indices].copy()

# Save the filtered data to a new CSV file
filtered_df.to_csv('filtered_message.csv', index=False)

Text length: 3, included: True
Text length: 19, included: True
Text length: 107, included: True
Text length: 3, included: True
Text length: 13, included: True
Text length: 42, included: True
Text length: 57, included: True
Text length: 15, included: True
Text length: 7, included: True
Text length: 15, included: True
Text length: 62, included: True
Text length: 7, included: True
Text length: 6, included: True
Text length: 23, included: True
Text length: 71, included: True
Text length: 10, included: True
Text length: 75, included: True
Text length: 8, included: True
Text length: 5, included: True
Text length: 12, included: True
Text length: 30, included: True
Text length: 5, included: True
Text length: 7, included: True
Text length: 61, included: True
Text length: 68, included: True
Text length: 9, included: True
Text length: 32, included: True
Text length: 15, included: True
Text length: 22, included: True
Text length: 16, included: True
Text length: 10, included: True
Text length: 80, 

In [ ]:
import pandas as pd
from transformers import pipeline

# Read the filtered CSV file
filtered_df = pd.read_csv('filtered_message.csv')

# Extract data from "Content" column
filtered_content_data = filtered_df['Content'].tolist()

# Define the sentiment analysis model to use
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

try:
    # Load the sentiment analysis pipeline
    sentiment_pipeline = pipeline("sentiment-analysis", model=model_name)

    # Perform sentiment analysis
    results = sentiment_pipeline(filtered_content_data)

    # Extract labels and scores
    labels = [result['label'] for result in results]
    scores = [result['score'] for result in results]

    # Add results to the dataframe
    filtered_df['Label'] = labels
    filtered_df['Score'] = scores

    # Save results to a new CSV file
    filtered_df.to_csv('message_with_sentiment.csv', index=False)

    # Output to verify
    print(filtered_df.head())
except Exception as e:
    print(f"An error occurred with model {model_name}: {e}")


Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


       AuthorID              Author                 Date  \
0  1.460000e+17           chartguy.  01/01/2024 05:56 AM   
1  1.030000e+18  toxicwaifu.dcl.eth  01/01/2024 08:01 AM   
2  8.740000e+17        cheddarqueso  01/01/2024 08:23 AM   
3  8.580000e+17      help_center017  01/01/2024 12:11 PM   
4  8.580000e+17      help_center017  01/01/2024 12:11 PM   

                                             Content Attachments  \
0                                                 Hi         NaN   
1                Happy New Year from Germany â¨ð         NaN   
2  Happy New Year DCL fam and team. I am thankful...         NaN   
3                                            Welcome         NaN   
4                                Happy new year ð         NaN   

             Reactions     Label     Score  
0                  NaN   neutral  0.464066  
1                  NaN  positive  0.941503  
2  ð (2),â¤ï¸ (1)  positive  0.988165  
3                  NaN  positive  0.726693  
4    

## Daily Average Sentiment

In [ ]:
import pandas as pd

# Read data from file
file_path = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/Data/message_with_sentiment.csv'  # Replace with your CSV file path
df = pd.read_csv(file_path)

# Check column names
print(df.columns)

# Define function to calculate sentiment score
def calculate_sentiment_score(label, score):
    if label == 'positive':
        return score
    elif label == 'negative':
        return -score
    else:
        return 0

# Calculate sentiment scores
df['Sentiment_Score'] = df.apply(lambda row: calculate_sentiment_score(row['Label'], row['Score']), axis=1)

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date']).dt.date

# Calculate daily average sentiment scores
daily_sentiment = df.groupby('Date')['Sentiment_Score'].mean().reset_index()

# Define labeling
def label_sentiment(score):
    if score > 0.1:  # Define threshold for positive sentiment
        return 'positive'
    elif score < -0.1:  # Define threshold for negative sentiment
        return 'negative'
    else:
        return 'neutral'

# Assign labels to daily average sentiment scores
daily_sentiment['Label'] = daily_sentiment['Sentiment_Score'].apply(label_sentiment)

# Save results to a new CSV file
output_file_path = 'daily_sentiment_scores.csv'  # Replace with the path where you want to save it
daily_sentiment.to_csv(output_file_path, index=False)

# Output to verify
print(daily_sentiment.head())


Index(['AuthorID', 'Author', 'Date', 'Content', 'Attachments', 'Reactions',
       'Label', 'Score'],
      dtype='object')
         Date  Sentiment_Score     Label
0  2024-01-01         0.871314  positive
1  2024-01-02         0.431091  positive
2  2024-01-03         0.070448   neutral
3  2024-01-04         0.208807  positive
4  2024-01-05         0.036255   neutral


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# 定义CSV文件的路径
file_path = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/Data/message_with_sentiment.csv'

# 读取CSV文件
data = pd.read_csv(file_path)

# 确保日期列是datetime格式
data['Date'] = pd.to_datetime(data['Date'], format='%m/%d/%Y %I:%M %p')

# 设置日期范围
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 6, 14)
date_range = pd.date_range(start_date, end_date)

# 检查缺失日期
existing_dates = data['Date'].dt.date.unique()
missing_dates = set(date_range.date) - set(existing_dates)

print(f"Missing dates: {sorted(missing_dates)}")


Missing dates: [datetime.date(2024, 4, 28), datetime.date(2024, 5, 13)]


In [ ]:
import pandas as pd
from datetime import datetime

# 定义CSV文件的路径
file_path = 'https://raw.githubusercontent.com/Xintong1122/KDD/main/Data/daily_sentiment_scores.csv'

# 读取CSV文件
data = pd.read_csv(file_path)

# 确保日期列是datetime格式
data['Date'] = pd.to_datetime(data['Date'], format='%Y-%m-%d')

# 补充缺失日期的数据
missing_dates = [datetime(2024, 4, 28), datetime(2024, 5, 13)]
new_rows = [{'Date': date, 'Sentiment_Score': 0.0, 'Label': 'neutral'} for date in missing_dates]

# 使用pd.concat方法来添加新行
data = pd.concat([data, pd.DataFrame(new_rows)], ignore_index=True)

# 保留score的小数点后四位
data['Sentiment_Score'] = data['Sentiment_Score'].round(4)

# 按日期排序
data = data.sort_values(by='Date')

# 保存到新的CSV文件
new_file_path = 'daily_processed_scores.csv'
data.to_csv(new_file_path, index=False)

